In [1]:
%pip install wikipedia
%pip install pyvis

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: pyvis in c:\users\kiit0001\appdata\local\programs\python\python312\lib\site-packages (0.3.2)




[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import spacy
import pandas as pd
import networkx as nx
from pyvis.network import Network
import wikipedia
import re
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\KIIT0001\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Get Wikipedia topic
topic = input("Enter Wikipedia topic: ")
wiki_page = wikipedia.page(topic)
text = wiki_page.content

# Clean text
text = re.sub(r'==.*?==+', '', text)
text = text.replace('\n', ' ')

In [5]:
# Extract entity pairs
def extract_entities(sentence):
    doc = nlp(sentence)
    subject = ""
    obj = ""
    for token in doc:
        if "subj" in token.dep_:
            subject = token.text
        if "obj" in token.dep_:
            obj = token.text
    return subject, obj

sentences = [sent.text for sent in nlp(text).sents]

data = []
for sentence in sentences:
    subj, obj = extract_entities(sentence)
    if subj and obj:
        data.append((subj, obj, sentence))

kg_df = pd.DataFrame(data, columns=["source", "target", "relation"])
print(kg_df)

          source       target  \
0    Paracetamol        fever   
1             It      Tylenol   
2    Paracetamol     migraine   
3       benefits      origins   
4           pain         them   
..           ...          ...   
223     efficacy  formulation   
224   ulceration         dogs   
225    treatment    ingestion   
226  Paracetamol         Guam   
227         bait       snakes   

                                              relation  
0    Paracetamol, or acetaminophen, is an analgesic...  
1    It is a widely available over-the-counter drug...  
2    Paracetamol relieves pain in both acute mild m...  
3    At a standard dose, paracetamol slightly reduc...  
4    The aspirin/paracetamol/caffeine combination a...  
..                                                 ...  
223  Paracetamol is considered a weak analgesic and...  
224  The main effect of toxicity in dogs is liver d...  
225  Acetylcysteine treatment is efficacious in dog...  
226  Paracetamol is lethal to sna

In [5]:
# ------------------ CLEAN GRAPH ------------------

# Remove stopwords
kg_df = kg_df[~kg_df["source"].str.lower().isin(stop_words)]
kg_df = kg_df[~kg_df["target"].str.lower().isin(stop_words)]

# Remove duplicates & self loops
kg_df = kg_df.drop_duplicates()
kg_df = kg_df[kg_df["source"] != kg_df["target"]]

# ------------------ BUILD GRAPH ------------------

G = nx.from_pandas_edgelist(
    kg_df,
    "source",
    "target",
    create_using=nx.DiGraph()
)

# ------------------ GRAPH ANALYTICS ------------------

degree_centrality = nx.degree_centrality(G)
pagerank_scores = nx.pagerank(G)

# Get top 60 nodes by PageRank
top_nodes = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:60]
important_set = set(node for node, _ in top_nodes)

# ------------------ EDGE-BASED FILTERING ------------------

edges_to_keep = []

for u, v in G.edges():
    if u in important_set or v in important_set:
        edges_to_keep.append((u, v))

G_filtered = nx.DiGraph()
G_filtered.add_edges_from(edges_to_keep)

# ------------------ PRINT ANALYTICS ------------------

print("\nTop 5 Entities by Degree Centrality:")
for node, score in sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(node, score)

print("\nTop 5 Entities by PageRank:")
for node, score in sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(node, score)



Top 5 Entities by Degree Centrality:
music 0.057777777777777775
rappers 0.057777777777777775
rap 0.04
MC 0.03111111111111111
lyrics 0.03111111111111111

Top 5 Entities by PageRank:
music 0.031710208663215465
racism 0.016188926162564763
Machine 0.016188926162564763
flow 0.013077161933297794
lyrics 0.011925322430132267


In [6]:
from pyvis.network import Network
import os

def visualize_pyvis(G_filtered, pagerank_scores, top_nodes):

    # Create network with dark theme
    net = Network(
        height="750px",
        width="100%",
        bgcolor="#111111",
        font_color="white",
        directed=True
    )

    most_important = top_nodes[0][0]

    # Normalize PageRank for better size scaling
    max_pr = max(pagerank_scores.values())
    min_pr = min(pagerank_scores.values())

    def scale_size(pr):
        return 12 + ((pr - min_pr) / (max_pr - min_pr)) * 25

    # Add nodes
    for node in G_filtered.nodes():

        if node in pagerank_scores:
            size = scale_size(pagerank_scores[node])
        else:
            size = 12

        if node == most_important:
            net.add_node(
                node,
                label=node,
                size=45,
                color="#ff4d4d",
                font={"color": "white"}
            )
        else:
            net.add_node(
                node,
                label=node,
                size=size,
                color="#1f77ff",
                font={"color": "white"}
            )

    # Add edges
    for u, v in G_filtered.edges():

        if u == most_important or v == most_important:
            net.add_edge(u, v, width=4, color="#ffffff")
        else:
            net.add_edge(u, v, width=2, color="#888888")

    # Improve spacing
    net.barnes_hut(
        gravity=-8000,
        central_gravity=0.3,
        spring_length=150,
        spring_strength=0.05,
        damping=0.09
    )

    # Save file
    output_file = "knowledge_graph_dark.html"
    net.write_html(output_file)

    print("Graph saved at:", os.path.abspath(output_file))
visualize_pyvis(G_filtered, pagerank_scores, top_nodes)

Graph saved at: c:\Users\KIIT0001\projects\RES PROJECTS\knowledge-graph-explorer\knowledge_graph_dark.html


In [ ]:
def query_entity(G):
    if G.number_of_nodes() == 0:
        print("Graph is empty. No entities to query.")
        return

    print("\n--- Knowledge Graph Query Engine ---")
    print("Type an entity name to explore its connections.")
    print("Type 'list' to see some available entities.")
    print("Type 'exit' to quit.\n")

    # Create lowercase mapping for case-insensitive search
    node_map = {node.lower(): node for node in G.nodes()}

    while True:
        user_input = input("Enter entity: ").strip().lower()

        if user_input == "exit":
            print("Exiting query engine.")
            break

        if user_input == "list":
            print("\nSample Available Entities:")
            for node in list(G.nodes())[:20]:
                print("-", node)
            print()
            continue

        if user_input in node_map:
            actual_node = node_map[user_input]

            print(f"\nConnections for '{actual_node}':\n")

            # Outgoing edges
            outgoing = list(G.successors(actual_node))
            incoming = list(G.predecessors(actual_node))

            if outgoing:
                print("→ Outgoing Relations:")
                for neighbor in outgoing:
                    print("   ", actual_node, "→", neighbor)
            else:
                print("→ No outgoing relations.")

            if incoming:
                print("\n← Incoming Relations:")
                for neighbor in incoming:
                    print("   ", neighbor, "→", actual_node)
            else:
                print("\n← No incoming relations.")

            print()

        else:
            print("Entity not found. Type 'list' to see available entities.\n")
query_entity(G)


--- Knowledge Graph Query Engine ---
Type an entity name to explore its connections.
Type 'list' to see some available entities.
Type 'exit' to quit.

Entity not found. Type 'list' to see available entities.

Exiting query engine.


: 